# 8 — Regex : lire, extraire et valider des motifs


## Fil conducteur

Après les techniques algorithmiques générales, ce chapitre présente un outil
spécialisé pour le texte. Le même raisonnement reste valable : partir d'un contrat,
construire une solution par petites règles, connaître ses limites et tester les
faux positifs. Ce module peut être traité après les graphes sans casser la progression.


## 1. Une description de texte
Une expression régulière décrit un motif de caractères. `chat` cherche ce texte
littéral ; `[0-9]+` cherche une ou plusieurs occurrences de chiffres ASCII.
Le module `re` fournit un moteur qui essaie de faire correspondre ce motif au texte.
Une regex est utile pour des formats textuels ; une boucle, `split` ou une méthode
comme `startswith` peut être plus lisible pour une tâche simple.

### Exemple illustré — un motif décrit ce qu’il faut repérer

```text
texte  : commande A12, puis B7
motif  : [0-9]+
trouvé :           ^^       ^
```

`[0-9]+` signifie « une ou plusieurs occurrences d’un chiffre ASCII ». Les zones
soulignées sont `12` et `7`.


## 2. Deux langages superposés
Python lit d'abord la chaîne, puis le moteur regex lit son contenu. Écris généralement
les motifs avec une chaîne brute : `r"\d+"`. Le préfixe r évite que Python
interprète la plupart des antislashs. Il ne désactive **pas** leur sens pour la regex.
Pour un point littéral, `r"\."` ; `.` seul signifie un caractère quelconque,
sauf le saut de ligne avec les options par défaut.

### Exemple illustré — Python lit la chaîne avant le moteur regex

| Écriture Python | Caractères reçus par le moteur |
|---|---|
| `r"\d+"` | barre oblique, `d`, `+` |
| `"\\d+"` | barre oblique, `d`, `+` |

La chaîne brute préfixée par `r` évite de doubler la barre oblique dans beaucoup de
motifs.




*Python construit la chaîne ; le moteur interprète le motif ; l’API fournit le résultat.*


## 3. Une grammaire par petites briques
| Motif | Sens |
|---|---|
| `[abc]` | Un caractère parmi a, b, c |
| `[a-z]` | Une minuscule ASCII |
| `[^0-9]` | Un caractère autre qu’un chiffre ASCII |
| `\d` | Un chiffre décimal Unicode par défaut |
| `\w` | Un caractère alphanumérique Unicode ou `_` par défaut |
| `\s` | Un caractère d’espacement |
| `*` / `+` / `?` | Respectivement 0+, 1+, 0 ou 1 fois |
| `{2}` / `{2,4}` | Exactement 2 / entre 2 et 4 fois |
| `(ab)` | Groupe capturant la séquence ab |
| `(?:ab)` | Groupe sans capture |
| `a\|b` | Alternative a ou b |
| `^` / `$` | Début / fin, avec particularités selon les options |

Un quantificateur porte sur l'élément **juste avant** : `ab+` accepte ab, abb,
abbb ; `(ab)+` accepte ab, abab. `\d` n'est pas strictement `[0-9]` en mode Unicode.
Utilise `[0-9]` quand le contrat exige les dix chiffres ASCII.

### Exemple illustré — lire les briques de gauche à droite

```text
A [0-9] {2}
│   │     └─ exactement deux fois
│   └─────── un chiffre ASCII
└─────────── la lettre A
```

Le motif accepte `A12`, mais pas `A7` ni `B12`. Le quantificateur `{2}` porte sur
la brique placée juste avant lui.




*Chaque morceau du motif traduit une règle précise du contrat, de gauche à droite.*

### Un modèle mental du moteur

Le moteur essaie de faire avancer une position dans le texte et une position dans
le motif. Une alternative ou un quantificateur crée plusieurs possibilités. Si un
choix mène plus tard à un échec, un moteur à retour arrière peut revenir au dernier
choix et essayer une autre possibilité. La gourmandise indique l'ordre des essais,
pas une garantie de résultat « plus correct ».

Ce modèle explique deux phénomènes : un motif gourmand peut d'abord consommer trop
loin puis reculer, et des répétitions ambiguës imbriquées peuvent réexaminer les
mêmes portions énormément de fois. Il n'est pas nécessaire de connaître toute la
construction d'un automate pour écrire les motifs du cours, mais il faut éviter de
considérer une regex comme une simple boucle toujours linéaire.


## 4. L’API dépend de la question
- `search` : existe-t-il une occurrence quelque part ? Rend un Match ou None.
- `match` : une occurrence commence-t-elle au début ? Ne valide pas forcément tout.
- `fullmatch` : la chaîne entière respecte-t-elle le motif ? À privilégier pour valider.
- `findall` : obtenir les occurrences ; des groupes capturants changent la forme du résultat.
- `finditer` : parcourir les Match avec leurs positions, sans matérialiser toute une liste.
- `sub` : remplacer les occurrences.

Un Match n'est pas directement la chaîne trouvée : `.group()` récupère le texte,
`.span()` récupère le début et la fin (borne finale exclue). Vérifie qu'il n'est pas
None avant d'appeler ces méthodes. `^...$` et `fullmatch` ne sont pas toujours
interchangeables : `$` peut correspondre juste avant un saut de ligne final.

### Exemple illustré — choisir l’API à partir de la question

```text
une occurrence existe ? ──→ search
tout le texte est valide ? → fullmatch
toutes les occurrences ? ─→ findall / finditer
remplacer ? ───────────────→ sub
```

Le motif peut rester identique ; c’est la question posée au texte qui change la
fonction appelée.


### Même texte, questions différentes, fonctions différentes

Avec le motif `[0-9]+` et le texte `"A12 puis B7"` :

| Question | Appel | Résultat simplifié |
|---|---|---|
| existe-t-il un nombre ? | `search` | premier Match : `12` |
| tout le texte est-il un nombre ? | `fullmatch` | `None` |
| quels sont tous les nombres ? | `findall` | `['12', '7']` |
| où sont-ils ? | `finditer` | Matchs `(1,3)` et `(10,11)` |
| les masquer | `sub('#', texte)` | `"A# puis B#"` |

Le motif décrit **quoi reconnaître**. La fonction choisie décrit **ce que l’on veut
faire** avec les correspondances.

## 5. Gourmand, minimal et contraintes
`<.*>` sur `<a> texte <b>` peut absorber tout le texte jusqu'au dernier >.
`<.*?>` essaie d'abord la correspondance la plus courte. `[^>]*` exprime directement
« autant de caractères que possible sauf > ». Ces motifs servent ici d'exemple ;
ils ne constituent pas un parseur HTML. Des structures imbriquées demandent
souvent un parseur adapté.

### Exemple illustré — gourmand contre minimal

Texte : `<b>x</b><i>y</i>`

| Motif simplifié | Zone choisie |
|---|---|
| `<.*>` gourmand | tout, du premier `<` au dernier `>` |
| `<.*?>` minimal | `<b>`, puis les balises suivantes séparément |

« Minimal » signifie s’arrêter dès qu’une correspondance complète devient possible.




*Image — Minimal s’arrête dès qu’une correspondance est possible.*


## 6. Valider une forme ne valide pas le sens
`[0-9]{4}-[0-9]{2}-[0-9]{2}` vérifie une forme de date. Il accepte 2025-99-99.
Pour vérifier une vraie date, ajoute `datetime.date.fromisoformat`.
Une regex simplifiée d'adresse e-mail ne prouve ni l'existence ni la validité complète
selon tous les standards. Dans ce cours, on utilise des contrats volontairement précis.

### Exemple illustré — une date peut avoir la bonne forme et le mauvais sens

| Texte | Forme `AAAA-MM-JJ` | Date réelle |
|---|:---:|:---:|
| `2024-02-29` | oui | oui |
| `2024-99-99` | oui | non |

La regex peut contrôler les positions des chiffres et tirets. Un calendrier doit
ensuite vérifier le sens des nombres.




*Image — Une validation métier complète la regex.*


## 7. Complexité et retour arrière du moteur
N'affirme pas « toutes les regex sont O(n) ». Le motif et le moteur influencent
le coût. Certaines répétitions ambiguës imbriquées, comme `(a+)+$`, peuvent explorer
énormément de partitions quand la correspondance échoue. C'est un autre usage
du retour arrière que les exercices d'énumération du chapitre précédent.
Réduis l'ambiguïté, évite les motifs inutilement complexes et teste les échecs.
Ici on n'exécute pas de démonstration potentiellement très lente.

**Références primaires :** [Documentation Python re](https://docs.python.org/3/library/re.html)
et [Regex HOWTO](https://docs.python.org/3/howto/regex.html).

### Exemple illustré — plusieurs découpages peuvent être essayés

Pour un motif ambigu comme `(a+)+b` sur une longue suite de `a` sans `b` :

```text
aaaa = aaaa
     = aaa + a
     = aa + aa
     = a + a + aa   …
```

Un moteur à retour arrière peut tester de nombreux découpages avant d’échouer. Des
quantificateurs imbriqués demandent donc une attention particulière.




*Image — Le retour arrière peut explorer beaucoup de possibilités.*


## 8. Construire un motif à partir du contrat

Une méthode fiable consiste à traduire une règle à la fois.

1. Écrire des exemples qui doivent réussir **et** échouer.
2. Découper le format en éléments : alphabet autorisé, ordre, répétitions et
   séparateurs littéraux.
3. Choisir l'API : chercher une occurrence n'est pas valider toute la chaîne.
4. Utiliser des groupes uniquement si leur structure ou leur contenu est utile.
5. Confier les règles sémantiques à du code spécialisé : date, nombre, parseur.
6. Tester aussi les sauts de ligne, espaces, caractères Unicode et entrées longues.

N'utilise pas une regex lorsque `split`, `startswith` ou une comparaison directe
exprime mieux le contrat. Une regex n'est pas non plus un parseur général pour une
syntaxe récursive comme du HTML imbriqué, ni une preuve qu'une adresse ou une
ressource existe réellement.

### Exemple illustré — traduire le contrat morceau par morceau

Contrat : une lettre, puis 2 à 4 lettres ou chiffres, et rien d’autre.

| Règle | Fragment |
|---|---|
| première lettre | `[A-Za-z]` |
| 2 à 4 caractères autorisés | `[A-Za-z0-9]{2,4}` |
| chaîne entière | `fullmatch` |

Motif final : `[A-Za-z][A-Za-z0-9]{2,4}`.


## Extraire, valider, remplacer

Prédis la différence entre search et fullmatch.


In [1]:
import re
texte = "Commande 42, lot 007"
match = re.search(r"[0-9]+", texte)
assert match is not None
print(match.group(), match.span())
print(re.findall(r"[0-9]+", texte))
assert re.search(r"[0-9]+", "abc42") is not None
assert re.fullmatch(r"[0-9]+", "abc42") is None
assert re.fullmatch(r"[0-9]+", "42") is not None
print(re.sub(r"[0-9]+", "#", texte))


42 (9, 11)
['42', '007']
Commande #, lot #


`findall` renvoie ici des chaînes et garde les zéros initiaux. Convertir en int ferait perdre ce format. Les motifs sont des descriptions textuelles, pas des nombres.


## Groupes et positions

Compare groupe capturant et groupe non capturant.


In [2]:
import re
texte = "ab abab"
print("capturant :", re.findall(r"(ab)+", texte))
print("non capturant :", re.findall(r"(?:ab)+", texte))
assert re.findall(r"(ab)+", texte) == ["ab", "ab"]
assert re.findall(r"(?:ab)+", texte) == ["ab", "abab"]
for m in re.finditer(r"[0-9]+", "A12 B7"):
    print(m.group(), m.span())


capturant : ['ab', 'ab']
non capturant : ['ab', 'abab']
12 (1, 3)
7 (5, 6)


Un groupe répété ne rend ici que sa dernière capture. Plusieurs groupes capturants font renvoyer des tuples à findall. Utilise finditer et group(0) si tu veux toujours la correspondance complète.


## Gourmandise et chiffres Unicode

Ces contre-exemples permettent de préciser le contrat.


In [3]:
import re
texte = "<a> texte <b>"
print(re.findall(r"<.*>", texte))
print(re.findall(r"<.*?>", texte))
print(re.findall(r"<[^>]*>", texte))
assert re.fullmatch(r"\d+", "١٢") is not None
assert re.fullmatch(r"[0-9]+", "١٢") is None
assert re.search(r"^[0-9]+$", "42\n") is not None
assert re.fullmatch(r"[0-9]+", "42\n") is None


['<a> texte <b>']
['<a>', '<b>']
['<a>', '<b>']


La regex doit refléter les règles du format, pas seulement fonctionner sur deux exemples heureux. Les trois motifs de balises restent des exemples pédagogiques simples, pas une solution générale au HTML.


## Exercices différenciés

Essaie chaque exercice avant de consulter le corrigé. Pour lancer les tests d’un exercice, remplace son `pass`, puis enlève le `#` devant les appels de test. Les cellules incomplètes restent exécutables.


### Palier A — Extraire les entiers ASCII

Renvoie les blocs de chiffres ASCII d’un texte sous forme de chaînes. Ignore les signes : -12 donne "12".

<details><summary>Indice progressif</summary>

Une classe suivie d’un quantificateur suffit.

</details>


In [4]:
def extraire_entiers(texte):
    pass


In [5]:
# assert extraire_entiers("A12 B007 -3") == ["12","007","3"]
# assert extraire_entiers("rien") == []
# assert extraire_entiers("١٢ 12") == ["12"]


### Palier B — Identifiant au contrat précis

Valide un identifiant de 3 à 12 caractères : première lettre ASCII, puis lettres ASCII, chiffres ou underscore. Pas d’espace, pas d’accent.

<details><summary>Indice progressif</summary>

La première lettre compte déjà pour 1 ; la suite doit en compter 2 à 11. Utilise fullmatch.

</details>


In [6]:
def identifiant_valide(texte):
    pass


In [7]:
# assert identifiant_valide("Ana_2")
# assert identifiant_valide("abc")
# assert not identifiant_valide("ab")
# assert not identifiant_valide("2abc")
# assert not identifiant_valide("élise")
# assert not identifiant_valide("abc\n")
# assert not identifiant_valide("a"*13)


### Palier C — Lire des lignes de log

Chaque ligne a exactement la forme YYYY-MM-DD LEVEL message, avec LEVEL parmi INFO, WARN, ERROR ; le message est non vide, commence et finit par un caractère non blanc. Renvoie un dict date/level/message. Lève ValueError si le format ou la date civile est invalide.

<details><summary>Indice progressif</summary>

Utilise des groupes nommés, fullmatch et date.fromisoformat. Vérifie séparément les espaces aux bords du message.

</details>


In [8]:
def lire_log(ligne):
    pass


In [9]:
# assert lire_log("2024-02-29 INFO Début") == {"date":"2024-02-29","level":"INFO","message":"Début"}
# for mauvais in ["2025-02-29 INFO X", "2024-01-01 DEBUG X", "2024-01-01 INFO ", "2024-01-01 INFO X\n", "2024-01-01 INFO  X"]:
#     try:
#         lire_log(mauvais)
#     except ValueError:
#         pass
#     else:
#         raise AssertionError(mauvais)


## Questions de compréhension

1. Pourquoi `search` ne valide-t-il pas un format complet ?
2. Quelle différence y a-t-il entre `[0-9]` et `\d` avec Python par défaut ?
3. Pourquoi une date conforme au motif peut-elle rester invalide ?
4. Quand une boucle, une méthode de chaîne ou un parseur est-il préférable ?


## À retenir et vérifier

Lis la regex brique par brique. Choisis l’API selon la question. Ajoute toujours des tests qui doivent échouer. Une forme valide ne garantit pas la validité sémantique.

**Autoévaluation :** peux-tu expliquer l'état des variables sur un petit exemple,
justifier la condition d'arrêt et donner un cas limite ? Peux-tu distinguer
l'espace de travail de l'espace occupé par les résultats ?

**Débrief en binôme :** présente une première solution correcte, puis un compromis
(vitesse, mémoire, lisibilité). Une optimisation doit préserver le contrat.


**Étape suivante.** Le chapitre 9 réunira toutes les techniques dans une méthode de résolution et de test.
